# 2. Merge Single-Cell Profiles

## Purpose
This notebook reads the per-compartment DuckDB produced by notebook 1 for a single
well-FOV and merges the Nuclei, Cell, and Cytoplasm tables into a single-cell (SC)
parquet profile. Organoid and Nucleocentric profiles are passed through and saved as
separate parquets.

This is **step 2 of Stage 4 (image-based profiling)**. It runs once per well-FOV and
is typically submitted as a child job via the SLURM scheduler.

## Inputs
- `data/{patient}/image_based_profiles/0.converted_profiles/{well_fov}/{well_fov}.duckdb`
  - Five compartment tables: `Organoid`, `Nuclei`, `Cell`, `Cytoplasm`, `Nucleocentric`
  - Produced by notebook 1 (`1.merge_feature_parquets.ipynb`)

## Outputs
Three parquet files written to `data/{patient}/image_based_profiles/0.converted_profiles/{well_fov}/`:

| File | Content | Rows |
|---|---|---|
| `sc_profiles_{well_fov}.parquet` | Merged Nuclei + Cell + Cytoplasm features | One row per object present in all three compartments |
| `organoid_profiles_{well_fov}.parquet` | Organoid features passed through | One row per segmented organoid |
| `nucleocentric_profiles_{well_fov}.parquet` | Nucleocentric features passed through | One row per nucleus-centered volume |

## Notes
- Only objects present in **all three** of Nuclei, Cell, and Cytoplasm are retained in the SC profile.
  Objects segmented in only some compartments are dropped.
- Object IDs are reassigned to a sequential `1..N` range at the end of this notebook.
  The original segmentation mask IDs are not preserved.

In [1]:
import os
import pathlib

import duckdb
import pandas as pd
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    well_fov = "C2-2"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
input_sqlite_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/{well_fov}.duckdb"
).resolve(strict=True)
destination_sc_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_profiles_{well_fov}.parquet"
).resolve()
destination_organoid_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_profiles_{well_fov}.parquet"
).resolve()
destination_nucleocentric_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/nucleocentric_profiles_{well_fov}.parquet"
).resolve()
destination_sc_parquet_file.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load all five compartment tables from the DuckDB produced by notebook 1.
with duckdb.connect(input_sqlite_file) as con:
    tables = con.execute("SHOW TABLES").fetchdf()
    print(tables)
    nuclei_table = con.sql("SELECT * FROM Nuclei").df()
    cells_table = con.sql("SELECT * FROM Cell").df()
    cytoplasm_table = con.sql("SELECT * FROM Cytoplasm").df()
    organoid_table = con.sql("SELECT * FROM Organoid").df()
    nucleocentric_table = con.sql("SELECT * FROM Nucleocentric").df()

            name
0           Cell
1      Cytoplasm
2         Nuclei
3  Nucleocentric
4       Organoid


In [ ]:
# Retain only objects that were successfully segmented in all three compartments.
# A nucleus without a matched cell/cytoplasm (or vice versa) is not a valid
# single-cell profile and is dropped here.
nuclei_id_set = set(nuclei_table["object_id"].to_list())
cells_id_set = set(cells_table["object_id"].to_list())
cytoplasm_id_set = set(cytoplasm_table["object_id"].to_list())

# find the intersection of the three sets
intersection_set = nuclei_id_set.intersection(cells_id_set, cytoplasm_id_set)

# keep only the rows in the three tables that are in the intersection set
nuclei_table = nuclei_table[nuclei_table["object_id"].isin(intersection_set)]
cells_table = cells_table[cells_table["object_id"].isin(intersection_set)]
cytoplasm_table = cytoplasm_table[cytoplasm_table["object_id"].isin(intersection_set)]

In [6]:
# Merge the three compartment tables into a single-cell dataframe.
# Because object_ids were already filtered to the intersection in the cell above,
# this LEFT JOIN is effectively an INNER JOIN — no NaN-filled rows will result.
with duckdb.connect() as con:
    con.register("nuclei", nuclei_table)
    con.register("cells", cells_table)
    con.register("cytoplasm", cytoplasm_table)
    # Merge them with SQL
    merged_df = con.execute("""
        SELECT *
        FROM nuclei
        LEFT JOIN cells USING (object_id)
        LEFT JOIN cytoplasm USING (object_id)
    """).df()

## Reorder object IDs

Original segmentation IDs are assigned per-FOV by the mask labeling step and are not
globally unique. They are replaced here with a clean sequential `1..N` index.
Note: the original mask IDs are not preserved — traceability back to the segmentation
mask requires the `image_set` column (identifying the FOV) plus positional knowledge
of which objects survived the intersection filter above.

In [7]:
# replace the object_id with a new unique ID
organoid_table["object_id"] = [i for i in range(1, organoid_table.shape[0] + 1)]
merged_df["object_id"] = [i for i in range(1, merged_df.shape[0] + 1)]
nucleocentric_table["object_id"] = [
    i for i in range(1, nucleocentric_table.shape[0] + 1)
]

In [8]:
# save the organoid data as parquet
print(f"Final organoid data shape: {organoid_table.shape}")
organoid_table.to_parquet(destination_organoid_parquet_file, index=False)
organoid_table.head()

Final organoid data shape: (1, 3337)


,object_id,image_set,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,Organoid_NoChannel_AreaSizeShape_MinY,...,Organoid_Mito_Texture_DifferenceEntropy-256-3,Organoid_Mito_Texture_DifferenceVariance-256-3,Organoid_Mito_Texture_Entropy-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation1-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation2-256-3,Organoid_Mito_Texture_InverseDifferenceMoment-256-3,Organoid_Mito_Texture_SumAverage-256-3,Organoid_Mito_Texture_SumEntropy-256-3,Organoid_Mito_Texture_SumVariance-256-3,Organoid_Mito_Texture_Variance-256-3
0,1,C2-2,893762.0,963.357654,730.347989,3.032268,1676997.0,767,1234,454,...,0.555395,0.003494,0.920656,-0.528743,0.694843,0.95198,5.34566,0.721711,604.09819,155.782665


In [9]:
print(f"Final merged single cell dataframe shape: {merged_df.shape}")
# save the sc data as parquet
merged_df.to_parquet(destination_sc_parquet_file, index=False)
merged_df.head()

Final merged single cell dataframe shape: (5, 10011)


,object_id,image_set,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_AreaSizeShape_CenterX,Nuclei_NoChannel_AreaSizeShape_CenterY,Nuclei_NoChannel_AreaSizeShape_CenterZ,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,Nuclei_NoChannel_AreaSizeShape_MinY,...,Cytoplasm_ER_Texture_DifferenceEntropy-256-3,Cytoplasm_ER_Texture_DifferenceVariance-256-3,Cytoplasm_ER_Texture_Entropy-256-3,Cytoplasm_ER_Texture_InformationMeasureOfCorrelation1-256-3,Cytoplasm_ER_Texture_InformationMeasureOfCorrelation2-256-3,Cytoplasm_ER_Texture_InverseDifferenceMoment-256-3,Cytoplasm_ER_Texture_SumAverage-256-3,Cytoplasm_ER_Texture_SumEntropy-256-3,Cytoplasm_ER_Texture_SumVariance-256-3,Cytoplasm_ER_Texture_Variance-256-3
0,1,C2-2,56267.0,922.307125,516.065562,3.291112,81081.0,873,972,460,...,0.088052,0.003846,0.121534,-0.503483,0.280077,0.994486,1.219201,0.096689,271.148506,74.467481
1,2,C2-2,127038.0,1142.749114,824.163825,2.527354,205020.0,1036,1237,743,...,0.069813,0.003858,0.086623,-0.310947,0.173905,0.995848,0.759884,0.074277,175.066989,53.638780
2,3,C2-2,53916.0,853.549559,668.456098,3.062301,71680.0,791,919,612,...,0.053543,0.003866,0.063461,-0.263245,0.131136,0.996788,0.562087,0.055949,113.350609,37.535818
3,4,C2-2,116073.0,890.781612,855.293479,2.914743,191520.0,790,1018,768,...,0.301519,0.003725,0.443225,-0.397226,0.440788,0.979065,6.120752,0.350427,2005.052048,543.932165
4,5,C2-2,19009.0,848.441528,584.013836,5.019464,27729.0,797,914,546,...,0.051061,0.003866,0.061719,-0.312122,0.145113,0.996904,0.441664,0.054122,70.685735,22.505548


In [10]:
print(f"Final nucleocentric dataframe shape: {nucleocentric_table.shape}")
# save the nucleocentric data as parquet
nucleocentric_table.to_parquet(destination_nucleocentric_parquet_file, index=False)
nucleocentric_table.head()

Final nucleocentric dataframe shape: (5, 3074)


,object_id,image_set,Nucleocentric_ER_CHAMMI75_Feature0,Nucleocentric_ER_CHAMMI75_Feature1,Nucleocentric_ER_CHAMMI75_Feature10,Nucleocentric_ER_CHAMMI75_Feature100,Nucleocentric_ER_CHAMMI75_Feature101,Nucleocentric_ER_CHAMMI75_Feature102,Nucleocentric_ER_CHAMMI75_Feature103,Nucleocentric_ER_CHAMMI75_Feature104,...,Nucleocentric_DNA_SAMMed3D_Feature90,Nucleocentric_DNA_SAMMed3D_Feature91,Nucleocentric_DNA_SAMMed3D_Feature92,Nucleocentric_DNA_SAMMed3D_Feature93,Nucleocentric_DNA_SAMMed3D_Feature94,Nucleocentric_DNA_SAMMed3D_Feature95,Nucleocentric_DNA_SAMMed3D_Feature96,Nucleocentric_DNA_SAMMed3D_Feature97,Nucleocentric_DNA_SAMMed3D_Feature98,Nucleocentric_DNA_SAMMed3D_Feature99
0,1,C2-2,1.044036,-1.747662,5.271373,1.652093,3.600009,-2.035951,-2.626941,-1.388177,...,-0.007857,-0.063534,0.022455,-0.010480,0.038982,-0.033405,-0.080638,0.201893,0.340384,0.228202
1,2,C2-2,-0.905200,-3.933524,3.074258,4.089850,0.752838,-1.324602,-3.136194,3.317343,...,-0.005876,-0.056268,0.111211,-0.011026,0.036573,0.001545,-0.013848,0.265339,0.341833,0.236851
2,3,C2-2,1.684066,-0.704785,1.909121,0.580475,1.572879,0.224192,-1.368444,0.905167,...,-0.006609,-0.082976,0.002251,-0.010423,0.019516,-0.009156,-0.033942,0.167631,0.315088,0.195704
3,4,C2-2,0.348349,-6.802246,4.346939,2.675132,0.084892,-1.273036,-2.115760,2.262069,...,-0.005933,-0.104830,0.050275,-0.010643,0.029749,0.042743,0.026819,0.224942,0.343378,0.221355
4,5,C2-2,1.700979,0.412655,2.405114,1.858082,2.119611,-2.610634,-1.505595,-0.855071,...,-0.007288,-0.100506,-0.157013,-0.010526,0.024236,0.002353,0.028942,0.035437,0.359154,0.111226
